# Learned Battery Consumption Model

Goal: given a flight state (airspeed, climb rate, bank angle, altitude, wind), predict the instantaneous power draw the battery would deliver, so it can estimate the energy cost of a planned maneuver/path before flying it.

Data: IDF_DS Fixed-Wing UAS Telemetry Benchmark (Zenodo 16992976, UPV, CC BY 4.0), 110 real autonomous flights over a fixed 3981m circuit, SpeedyBee F405/INAV stack. Real vbat (V) and amperage (A) at ~62Hz.

Framing: inputs are restricted to things known from a plan (speed, climb rate, bank angle, altitude, wind).

The goal is to use this model for using predicted power consumption in const functions in path planning problem for fixed wing airfrafts.

In [ ]:
# Setup: get the SpeedyBee telemetry data (re-download only if not already present in this runtime)
import os
if not os.path.exists("speedybee_data/Speedybee DataSet/processed/telemetry"):
  !wget -q --show-progress -O speedybee.zip "https://zenodo.org/records/16992976/files/Speedybee%20DataSet.zip?download=1"
  !unzip -q speedybee.zip -d speedybee_data
  print("Downloaded and extracted.")
else:
  print("Data already present in this runtime.")
import glob
laps = sorted(glob.glob("speedybee_data/Speedybee DataSet/processed/telemetry/Lap*.csv"))
print(f"{len(laps)} flight CSVs found.")

In [ ]:
import re
import pandas as pd


def _lap_num(p):
    return int(re.search(r'Lap(\d+)', p).group(1))


missing_wind = [
    _lap_num(p) for p in laps
    if not {'wind[0]', 'wind[1]', 'wind[2]'}.issubset(pd.read_csv(p, nrows=0).columns)
]

print(f"{len(missing_wind)} flights missing wind columns:", sorted(missing_wind))

In [ ]:
# Group lap FILES into PHYSICAL flights. Consecutive lap numbers are often segments of the
# same continuous flight (energyCumulative and time carry over from one file to the next)
# rather than independent flights- splitting those across train/val/test would leak the
# same physical flight (same battery, same conditions) across sets.
#
# Detect a new physical flight whenever energy/time do NOT continue from the previous lap file.
probe_cols = ['time (us)', 'energyCumulative (mAh)']
lap_nums_sorted = sorted(_lap_num(p) for p in laps)
path_by_num = {_lap_num(p): p for p in laps}

endpoints = {}
for n in lap_nums_sorted:
    d = pd.read_csv(path_by_num[n], usecols=probe_cols)
    endpoints[n] = (d['time (us)'].iloc[0], d['time (us)'].iloc[-1],
                    d['energyCumulative (mAh)'].iloc[0], d['energyCumulative (mAh)'].iloc[-1])

physical_flight_id = {}
current_id = 0
prev_n = None
for n in lap_nums_sorted:
    t0, t1, e0, e1 = endpoints[n]
    if prev_n is None:
        current_id = 0
    else:
        _, pt1, _, pe1 = endpoints[prev_n]
        is_continuation = (t0 > pt1) and (e0 >= pe1 - 5)  # small tolerance for rollover
        if not is_continuation:
            current_id += 1
    physical_flight_id[n] = current_id
    prev_n = n

n_groups = len(set(physical_flight_id.values()))
print(f"{len(lap_nums_sorted)} lap files grouped into {n_groups} physical flights "
      f"(previously treated as {len(lap_nums_sorted)} independent flights).")

In [ ]:
# Check per-flight vbat range across all flights to catch hardware-config outliers (not assumed, found by inspection)

def _vbat_range(p):
    d = pd.read_csv(p, usecols=['vbat (V)'])['vbat (V)']
    return (d.min(), d.max())


vbat_by_flight = {
    _lap_num(p): _vbat_range(p) for p in laps
    if 'vbat (V)' in pd.read_csv(p, nrows=0).columns
}

vbat_df = pd.DataFrame(vbat_by_flight, index=['vbat_min', 'vbat_max']).T.sort_values('vbat_max', ascending=False)
print(vbat_df[vbat_df['vbat_max'] > 20])

In [ ]:
def extract_features(path, flight_id):
    df = pd.read_csv(path, usecols=[
        'attitude[0]', 'attitude[1]', 'navVel[0]', 'navVel[1]', 'navVel[2]',
        'BaroAlt (cm)', 'wind[0]', 'wind[1]', 'wind[2]', 'vbat (V)', 'amperage (A)',
    ])

    out = pd.DataFrame()
    out['flight_id'] = [flight_id] * len(df)
    out['bank_deg'] = df['attitude[0]'] / 10.0
    out['pitch_deg'] = df['attitude[1]'] / 10.0
    out['groundspeed'] = (df['navVel[0]'] ** 2 + df['navVel[1]'] ** 2) ** 0.5 / 100.0
    out['climb_rate'] = df['navVel[2]'] / 100.0
    out['altitude_m'] = df['BaroAlt (cm)'] / 100.0
    out['wind_x'] = df['wind[0]'] / 100.0
    out['wind_y'] = df['wind[1]'] / 100.0
    out['wind_vert'] = df['wind[2]'] / 100.0
    out['power_w'] = df['vbat (V)'] * df['amperage (A)']

    return out

In [ ]:
# missing-wind schema gap + anomalous ~35V/8S battery pack
EXCLUDE = {33, 34, 35, 36, 37, 38, 39, 40, 53, 54, 55, 56, 57} | set(missing_wind)

frames = [extract_features(p, physical_flight_id[_lap_num(p)]) for p in laps if _lap_num(p) not in EXCLUDE]
data = pd.concat(frames, ignore_index=True)

print(f"Used {len(frames)} lap files, skipped {len(EXCLUDE)}: {sorted(EXCLUDE)}")
print(f"Rows span {data['flight_id'].nunique()} distinct physical flights")
print(data.shape)
data.describe()

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import torch

np.random.seed(42)

flight_ids = sorted(data['flight_id'].unique())
train_ids, temp_ids = train_test_split(flight_ids, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

feature_cols = ['bank_deg', 'pitch_deg', 'groundspeed', 'climb_rate', 'altitude_m', 'wind_x', 'wind_y', 'wind_vert']

train_df = data[data['flight_id'].isin(train_ids)]
val_df = data[data['flight_id'].isin(val_ids)]
test_df = data[data['flight_id'].isin(test_ids)]

print(f'Train flights: {len(train_ids)}, rows: {len(train_df)}')
print(f'Val flights: {len(val_ids)}, rows: {len(val_df)}')
print(f'Test flights: {len(test_ids)}, rows: {len(test_df)}')

mu = train_df[feature_cols].mean()
sigma = train_df[feature_cols].std()

X_train = torch.tensor(((train_df[feature_cols] - mu) / sigma).values, dtype=torch.float32)
y_train = torch.tensor(train_df['power_w'].values, dtype=torch.float32).unsqueeze(1)
X_val = torch.tensor(((val_df[feature_cols] - mu) / sigma).values, dtype=torch.float32)
y_val = torch.tensor(val_df['power_w'].values, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(((test_df[feature_cols] - mu) / sigma).values, dtype=torch.float32)
y_test = torch.tensor(test_df['power_w'].values, dtype=torch.float32).unsqueeze(1)

baseline_pred = y_train.mean().item()
baseline_mae = (y_test - baseline_pred).abs().mean().item()
print(f'Naive baseline (predict train mean power = {baseline_pred:.2f}W): test MAE = {baseline_mae:.3f} W')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class PowerMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model = PowerMLP(len(feature_cols)).to(device)
Xtr, ytr = X_train.to(device), y_train.to(device)
Xv, yv = X_val.to(device), y_val.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

n = Xtr.shape[0]
batch_size = 4096
epochs = 30

train_losses = []
val_losses = []
best_val = float('inf')
best_state = None

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(n, device=device)
    total_loss = 0.0

    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = Xtr[idx], ytr[idx]

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.shape[0]

    train_loss = total_loss / n

    model.eval()
    with torch.no_grad():
        val_pred = model(Xv)
        val_loss = loss_fn(val_pred, yv).item()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch+1}/{epochs}  train_loss={train_loss:.2f}  val_loss={val_loss:.2f}')

model.load_state_dict(best_state)
print('Training complete. Best val loss:', best_val)

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test.to(device))

test_mae = (test_pred - y_test.to(device)).abs().mean().item()
improvement = (1 - test_mae / baseline_mae) * 100

print(f'MLP test MAE: {test_mae:.3f} W')
print(f'Naive baseline test MAE: {baseline_mae:.3f} W')
print(f'Improvement over baseline: {improvement:.1f}%')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train loss (MSE)')
plt.plot(val_losses, label='Val loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (W^2)')
plt.title('Training curves: PowerMLP')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

y_test_np = y_test.numpy().flatten()
test_pred_np = test_pred.cpu().numpy().flatten()

r2 = r2_score(y_test_np, test_pred_np)
mean_power = y_test_np.mean()
relative_error = test_mae / mean_power * 100

print(f'R^2 (variance explained): {r2:.4f}')
print(f'Mean true power in test set: {mean_power:.2f} W')
print(f'MLP MAE as % of mean power: {relative_error:.1f}%')

print()
print('Sample predictions (true vs predicted power, Watts):')

rng = np.random.default_rng(0)
idx_sample = rng.choice(len(y_test_np), size=10, replace=False)
for i in idx_sample:
    print(f'  true={y_test_np[i]:7.2f}W   pred={test_pred_np[i]:7.2f}W   diff={test_pred_np[i]-y_test_np[i]:+6.2f}W')

In [ ]:
test_df_reset = test_df.reset_index(drop=True)
test_df_reset['residual'] = test_pred_np - y_test_np

per_flight = test_df_reset.groupby('flight_id')['residual'].agg(['mean', 'std', 'count'])
per_flight = per_flight.sort_values('mean')

print('Per-flight residual (pred - true), test set:')
print(per_flight)
print()
print(f"Std of per-flight MEAN residual (between-flight bias): {per_flight['mean'].std():.2f} W")
print(f"Average within-flight residual std:                    {per_flight['std'].mean():.2f} W")

## Cross-validation over physical flights

The single train/val/test split above holds out only 4 physical flights for testing: a small, noisy sample. Before chasing feature or model changes, check how much the R²/MAE numbers actually vary depending on which flights end up in the test set, using 5-fold cross-validation grouped by physical flight (never splitting a physical flight across folds).

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


def train_eval_fold(train_flight_ids, test_flight_ids, epochs=30, batch_size=4096, seed=42):
    torch.manual_seed(seed)
    train_df_f = data[data['flight_id'].isin(train_flight_ids)]
    test_df_f = data[data['flight_id'].isin(test_flight_ids)]

    mu_f = train_df_f[feature_cols].mean()
    sigma_f = train_df_f[feature_cols].std()

    Xtr = torch.tensor(((train_df_f[feature_cols] - mu_f) / sigma_f).values, dtype=torch.float32)
    ytr = torch.tensor(train_df_f['power_w'].values, dtype=torch.float32).unsqueeze(1)
    Xte = torch.tensor(((test_df_f[feature_cols] - mu_f) / sigma_f).values, dtype=torch.float32)
    yte_np = test_df_f['power_w'].values

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fold_model = PowerMLP(len(feature_cols)).to(device)
    Xtr, ytr = Xtr.to(device), ytr.to(device)
    optimizer = optim.Adam(fold_model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    n = Xtr.shape[0]
    for epoch in range(epochs):
        fold_model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = Xtr[idx], ytr[idx]
            optimizer.zero_grad()
            pred = fold_model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

    fold_model.eval()
    with torch.no_grad():
        pred_te = fold_model(Xte.to(device)).cpu().numpy().flatten()

    mae = np.abs(pred_te - yte_np).mean()
    r2 = r2_score(yte_np, pred_te)
    return mae, r2, len(train_df_f), len(test_df_f)


flight_ids_all = sorted(data['flight_id'].unique())
print(f"{len(flight_ids_all)} physical flights total, running 5-fold group CV (no early stopping, fixed 30 epochs/fold)")
print()

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold_i, (train_idx, test_idx) in enumerate(kf.split(flight_ids_all)):
    train_flights = [flight_ids_all[i] for i in train_idx]
    test_flights = [flight_ids_all[i] for i in test_idx]
    mae, r2, n_train_rows, n_test_rows = train_eval_fold(train_flights, test_flights)
    fold_results.append((fold_i, len(train_flights), len(test_flights), mae, r2))
    print(f"Fold {fold_i}: {len(train_flights)} train flights / {len(test_flights)} test flights "
          f"({n_train_rows} / {n_test_rows} rows)  ->  MAE={mae:.3f}W  R2={r2:.4f}")

maes = np.array([r[3] for r in fold_results])
r2s = np.array([r[4] for r in fold_results])
print()
print(f"MAE across folds: mean={maes.mean():.3f}W  std={maes.std():.3f}W  min={maes.min():.3f}W  max={maes.max():.3f}W")
print(f"R2  across folds: mean={r2s.mean():.4f}  std={r2s.std():.4f}  min={r2s.min():.4f}  max={r2s.max():.4f}")

## Feature ablations tried and ruled out

Two physically motivated features were tried on top of the 8-feature baseline and evaluated with the same grouped 5-fold CV, to see if either would push R² past the ~0.67 ceiling:

- **Explicit airspeed** (`groundspeed` and `wind_x`/`wind_y` combined into an airspeed vector/magnitude, since power depends on airspeed, not groundspeed): mean R² 0.6676 → 0.6838, MAE 12.177W → 12.185W. Improvement smaller than the fold-to-fold noise (±0.04) and one fold got worse, not a real effect.
- **Turn-load factor** (`1/cos(bank)`, capturing the nonlinear induced-drag cost of turning): mean R² 0.6676 → 0.6862, MAE 12.177W → 11.831W. Same story, driven almost entirely by a single fold, not a consistent gain.

**Conclusion from both:** the ~0.67 R² ceiling isn't due to the network missing an obvious static recombination of its existing inputs. That's useful negative evidence pointing toward the per-flight residual diagnostic below being the right lead: the dominant error source is within-flight noise, not a missing snapshot feature.

## Conclusion
 ~0.67 R² looks like a genuine ceiling for per-timestamp regression on flight-dynamics state alone, not an artifact of model choice or a missing static feature. Meaningfully exceeding it likely requires a sequence model (e.g. a small LSTM/1D-CNN over a short window of recent samples, to capture motor/ESC/prop response lag) or additional sensor channels not present in this dataset (throttle/PWM command, propeller RPM)